# WildTrace — 02. Embedding Visualization
Embed a set of individual images (from `ml/datasets/processed`), reduce with PCA to 2D, and color each point by individual — mirroring the deck's Figure 7 to show clusters.

In [ ]:
import sys
sys.path.insert(0, '.')
from pathlib import Path
import numpy as np
from PIL import Image

from ml.embedding.extract_embeddings import EmbeddingExtractor

root = Path('ml/datasets/processed')
ckpt = 'ml/reid/checkpoints/densenet121_triplet_best.pt'
ext = EmbeddingExtractor(checkpoint_path=ckpt, embedding_dim=512)

embs, labels = [], []
for lab in sorted(p for p in root.iterdir() if p.is_dir())[:6]:
    for img_path in list(lab.glob('*.jpg'))[:12] + list(lab.glob('*.png'))[:12]:
        with Image.open(img_path) as im:
            embs.append(ext.embed(im.convert('RGB')))
            labels.append(lab.name)
embs = np.vstack(embs)

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

pca = PCA(n_components=2).fit_transform(embs)
unique = sorted(set(labels))
colors = plt.cm.tab20(np.linspace(0, 1, len(unique)))

plt.figure(figsize=(8, 6))
for c, lab in zip(colors, unique):
    mask = np.array(labels) == lab
    plt.scatter(pca[mask, 0], pca[mask, 1], color=c, label=lab, s=40, alpha=0.7)
plt.title('PCA-reduced Re-ID embeddings, colored by individual')
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.legend(loc='best', fontsize=8)
plt.grid(alpha=0.3)
plt.show()

print('Clusters indicate separability of individuals in embedding space.')